# NB05 - Loss Landscape Visualization
## Background
The geometry of the loss landscape determines how well an optimizer finds generalizable solutions. Sharp minima correlate with poor generalization (Hochreiter & Schmidhuber, 1997; Keskar et al., 2017) — a network in a sharp minimum has high curvature in many directions, meaning tiny perturbations cause large loss changes. Flat minima, by contrast, correspond to robust solutions. Filter normalization (Li et al., 2018) enables visualizing high-dimensional loss surfaces by projecting onto two random directions in parameter space.

## What You'll Learn
- Filter normalization technique for loss landscape visualization
- Computing sharpness: largest eigenvalue of the Hessian (power iteration)
- 1D and 2D loss landscape plots for quadratic, Rosenbrock, and neural network losses
- Flat vs sharp minimum comparison

## Why This Matters
Sharpness-Aware Minimization (SAM) directly targets the flat minima that generalize better. Understanding loss landscape visualization lets you diagnose whether your model is in a sharp minimum and motivates regularization techniques like weight decay, noise, and larger batch sizes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable, Tuple

# ── 1D Loss Slice ─────────────────────────────────────────────────────────
def plot_1d_slice(fn: Callable, center: np.ndarray, direction: np.ndarray,
                  alpha_range=(-2, 2), n_points=200, ax=None, label=''):
    alphas = np.linspace(*alpha_range, n_points)
    losses = [fn(center + a * direction)[0] for a in alphas]
    if ax is None:
        fig, ax = plt.subplots()
    ax.plot(alphas, losses, label=label)
    return alphas, losses

# ── Filter normalization ───────────────────────────────────────────────────
def filter_normalize(direction: np.ndarray, reference: np.ndarray) -> np.ndarray:
    """Scale direction to have same norm as reference (Li et al., 2018)."""
    ref_norm = np.linalg.norm(reference)
    dir_norm = np.linalg.norm(direction)
    if dir_norm < 1e-10:
        return direction
    return direction * (ref_norm / dir_norm)

# ── 2D Loss Surface ────────────────────────────────────────────────────────
def loss_surface_2d(fn: Callable, center: np.ndarray,
                     dir1: np.ndarray, dir2: np.ndarray,
                     alpha_range=(-2, 2), n_grid=40):
    alphas = np.linspace(*alpha_range, n_grid)
    betas  = np.linspace(*alpha_range, n_grid)
    Z = np.zeros((n_grid, n_grid))
    for i, a in enumerate(alphas):
        for j, b in enumerate(betas):
            params = center + a * dir1 + b * dir2
            Z[i, j] = fn(params)[0]
    return alphas, betas, Z

# ── Test functions ─────────────────────────────────────────────────────────
def quadratic(x): return x[0]**2 + 10*x[1]**2, np.array([2*x[0], 20*x[1]])
def rosenbrock(x):
    a, b = 1.0, 100.0
    loss = (a-x[0])**2 + b*(x[1]-x[0]**2)**2
    return loss, np.array([-2*(a-x[0])-4*b*x[0]*(x[1]-x[0]**2), 2*b*(x[1]-x[0]**2)])

# ── 2D landscape for quadratic ─────────────────────────────────────────────
center = np.array([0.0, 0.0])
dir1 = np.array([1.0, 0.0])
dir2 = np.array([0.0, 1.0])

alphas, betas, Z_quad = loss_surface_2d(quadratic, center, dir1, dir2,
                                          alpha_range=(-2, 2), n_grid=50)
_, _, Z_rb = loss_surface_2d(rosenbrock, np.array([1.0, 1.0]), dir1, dir2,
                               alpha_range=(-1.5, 1.5), n_grid=50)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (Z, title) in zip(axes, [(Z_quad, 'Quadratic Loss Landscape'),
                                   (Z_rb, 'Rosenbrock Loss Landscape')]):
    A, B = np.meshgrid(alphas, betas)
    cf = ax.contourf(A, B, np.log1p(Z.T), levels=30, cmap='RdYlBu_r')
    ax.contour(A, B, np.log1p(Z.T), levels=30, colors='k', linewidths=0.3, alpha=0.4)
    plt.colorbar(cf, ax=ax, label='log(1 + loss)')
    ax.set_title(title); ax.set_xlabel('alpha (dir1)'); ax.set_ylabel('beta (dir2)')

plt.tight_layout()
plt.savefig(f'{BASE}/s30_05_landscape_2d.png', dpi=100, bbox_inches='tight')
plt.show()
print('2D loss landscapes plotted.')

In [ ]:
# ── Sharpness: power iteration on finite-difference Hessian ───────────────
def hessian_vec_product(fn: Callable, params: np.ndarray,
                          v: np.ndarray, eps: float = 1e-4) -> np.ndarray:
    """Hessian-vector product via finite differences."""
    _, g1 = fn(params + eps * v)
    _, g2 = fn(params - eps * v)
    return (g1 - g2) / (2 * eps)

def power_iteration_sharpness(fn: Callable, params: np.ndarray,
                                n_iters: int = 50) -> float:
    """Estimate largest Hessian eigenvalue (sharpness) via power iteration."""
    v = np.random.randn(*params.shape)
    v /= np.linalg.norm(v)
    eigenvalue = 0.0
    for _ in range(n_iters):
        Hv = hessian_vec_product(fn, params, v)
        eigenvalue = np.dot(v, Hv)
        v = Hv / (np.linalg.norm(Hv) + 1e-12)
    return eigenvalue

# Compare sharpness at flat vs sharp minima
np.random.seed(42)

# Sharp: conditioned quadratic minimum
def sharp_bowl(x):
    return 100*x[0]**2 + 100*x[1]**2, np.array([200*x[0], 200*x[1]])

def flat_bowl(x):
    return 0.1*x[0]**2 + 0.1*x[1]**2, np.array([0.2*x[0], 0.2*x[1]])

params = np.array([0.01, 0.01])  # Near minimum
sharp_sharpness = power_iteration_sharpness(sharp_bowl, params)
flat_sharpness  = power_iteration_sharpness(flat_bowl, params)

print('=== Sharpness Comparison ===')
print(f'Sharp bowl - max Hessian eigenvalue: {sharp_sharpness:.2f}')
print(f'Flat bowl  - max Hessian eigenvalue: {flat_sharpness:.2f}')
print()
print('Sharp minimum: large eigenvalue -> small perturbations cause large loss increases')
print('Flat minimum:  small eigenvalue -> robust to weight perturbations')
print()
print('Connection to generalization:')
print('  A flat minimum occupies more volume in weight space')
print('  -> More likely to stay in low-loss region after distributional shift')
print('  -> Better generalization (PAC-Bayes perspective)')

In [ ]:
# ── 1D slices through flat vs sharp minima ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

center_flat  = np.array([0.0, 0.0])
center_sharp = np.array([0.0, 0.0])
direction = np.array([1.0, 0.5]) / np.linalg.norm([1.0, 0.5])

for ax, (fn, title) in zip(axes, [
    (flat_bowl,  'Flat Minimum (generalization-friendly)'),
    (sharp_bowl, 'Sharp Minimum (generalization risk)'),
]):
    plot_1d_slice(fn, center_flat, direction, alpha_range=(-1, 1), ax=ax, label='Loss slice')
    ax.set_title(title)
    ax.set_xlabel('Step along direction'); ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)
    # Mark minimum
    ax.axvline(0, color='red', linestyle='--', alpha=0.7, label='Minimum')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{BASE}/s30_05_flat_vs_sharp.png', dpi=100, bbox_inches='tight')
plt.show()
print('Flat vs sharp minimum visualization complete.')